# M10 · Séparabilité linéaire et convexité — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m10_separabilite_convexite/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m10_separabilite_convexite/03_notebook.ipynb`, créée par `python tools/start_chapter.py M10`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M10.1 à M10.5, M10.9 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M10.12 | L'enveloppe convexe dans le plan : la chaîne monotone d'Andrew · `convex_hull_2d` | 🔨 | ★★ | 25 |
|  | M10.13 | Séparabilité par programmation linéaire, et marge géométrique · `is_linearly_separable, margin_of_separator` | 🔨 | ★★ | 25 |
| B | M10.14 | Prédire : n points aux labels tirés au hasard en dimension d sont-ils séparables ? | 🔮 | ★ | 10 |
|  | M10.15 | Deux nuages, leurs enveloppes convexes et un hyperplan séparateur avec sa marge | 🎨 | ★ | 15 |
| C | M10.16 | Chasse à l'erreur : un produit vectoriel de signe inversé, une marge non normalisée | 🐛 | ★ | 10 |
|  | M10.17 | Des tests de propriétés pour l'enveloppe convexe | 🛠️ | ★ | 15 |
|  | M10.18 | Défi : la plus grande marge possible entre deux nuages donnés, à 1 % près | 🏆 | ★★★ | 20 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M10")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : segments et convexes, enveloppe convexe et produit vectoriel, séparateur et marge, astuce du biais, récurrence sur une norme, activations.
- Écrire les trois fonctions de `mymath/convex.py` et les tester.
- Prédire la séparabilité de labels au hasard, dessiner une marge, chasser deux erreurs, écrire des tests de propriétés, trouver la plus grande marge.

**Rappel express.** Convexe : $(1 - t)\mathbf{a} + t\mathbf{b} \in C$ ; $\mathbf{u} \times \mathbf{v} = u_1v_2 - u_2v_1$, positif à gauche ; séparable $\iff$ $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \ge 1$ a une solution $\iff$ enveloppes disjointes ; $\tilde{\mathbf{x}} = (1, \mathbf{x})$, $\tilde{\mathbf{w}} = (b, \mathbf{w})$ ; marge $\min_i \frac{y_i(\mathbf{w} \cdot \mathbf{x}_i + b)}{\lVert \mathbf{w} \rVert}$, la plus grande $= \frac{1}{2}\operatorname{dist}(\operatorname{conv} A, \operatorname{conv} B)$ ; $\lVert \mathbf{v}_{k+1} \rVert^2 \le \lVert \mathbf{v}_k \rVert^2 + c \Rightarrow \lVert \mathbf{v}_k \rVert^2 \le \lVert \mathbf{v}_0 \rVert^2 + kc$.

## Partie 0 · Vérifier tes exercices papier (✏️ M10.1 à M10.5, 🔗 M10.9)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[2, -1]` pour un point ou un vecteur, une liste de points `[[1, 0], [-2, 3]]` ;
- `True` / `False`, des lettres entre guillemets, `"BD"`, ou un intervalle entre guillemets, `"]0 ; 4]"` (crochet tourné vers l'extérieur : borne exclue).

Les réponses pas encore remplies affichent ⏳.

**Ex M10.1 — Convexe ou non : segments, demi-plans, intersections**

In [ ]:
# Ex M10.1: the values you found on paper
answer_M10_1a = ...  # a) a list of 2 numbers
answer_M10_1b = ...  # b) a number
answer_M10_1c = ...  # c) letters, as text
answer_M10_1d = ...  # d) a list of 2 numbers
answer_M10_1e = ...  # e) an integer
answer_M10_1f = ...  # f) letters, as text
answer_M10_1g = ...  # g) a list of 2 numbers (1 decimal)

for letter, answer in zip("abcdefg", [answer_M10_1a, answer_M10_1b, answer_M10_1c, answer_M10_1d, answer_M10_1e, answer_M10_1f, answer_M10_1g]):
    wbm.check(f"M10.1{letter}", answer)

**Ex M10.2 — L'enveloppe convexe de huit points, à la main**

In [ ]:
# Ex M10.2: the values you found on paper
answer_M10_2a = ...  # a) an integer
answer_M10_2b = ...  # b) a list of points
answer_M10_2c = ...  # c) an integer
answer_M10_2d = ...  # d) a number
answer_M10_2e = ...  # e) a number
answer_M10_2f = ...  # f) True or False
answer_M10_2g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M10_2a, answer_M10_2b, answer_M10_2c, answer_M10_2d, answer_M10_2e, answer_M10_2f, answer_M10_2g]):
    wbm.check(f"M10.2{letter}", answer)

**Ex M10.3 — Une droite qui sépare deux petits nuages, et sa marge**

In [ ]:
# Ex M10.3: the values you found on paper
answer_M10_3a = ...  # a) a list of 6 numbers
answer_M10_3b = ...  # b) a number (4 decimals)
answer_M10_3c = ...  # c) a list of integers
answer_M10_3d = ...  # d) an interval
answer_M10_3e = ...  # e) a number
answer_M10_3f = ...  # f) a number (4 decimals)
answer_M10_3g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M10_3a, answer_M10_3b, answer_M10_3c, answer_M10_3d, answer_M10_3e, answer_M10_3f, answer_M10_3g]):
    wbm.check(f"M10.3{letter}", answer)

**Ex M10.4 — L'astuce du biais : une droite du plan devient un plan de ℝ³ passant par l'origine**

In [ ]:
# Ex M10.4: the values you found on paper
answer_M10_4a = ...  # a) a list of 3 numbers
answer_M10_4b = ...  # b) a list of 3 numbers
answer_M10_4c = ...  # c) a list of 3 numbers
answer_M10_4d = ...  # d) a number (4 decimals)
answer_M10_4e = ...  # e) a list of 2 numbers
answer_M10_4f = ...  # f) True or False
answer_M10_4g = ...  # g) a list of 3 numbers

for letter, answer in zip("abcdefg", [answer_M10_4a, answer_M10_4b, answer_M10_4c, answer_M10_4d, answer_M10_4e, answer_M10_4f, answer_M10_4g]):
    wbm.check(f"M10.4{letter}", answer)

**Ex M10.5 — Borner une norme pas à pas : ‖v_{k+1}‖² ≤ ‖v_k‖² + c**

In [ ]:
# Ex M10.5: the values you found on paper
answer_M10_5a = ...  # a) a number
answer_M10_5b = ...  # b) a number
answer_M10_5c = ...  # c) a number
answer_M10_5d = ...  # d) a number
answer_M10_5e = ...  # e) a number (2 decimals)
answer_M10_5f = ...  # f) an integer
answer_M10_5g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M10_5a, answer_M10_5b, answer_M10_5c, answer_M10_5d, answer_M10_5e, answer_M10_5f, answer_M10_5g]):
    wbm.check(f"M10.5{letter}", answer)

**Ex M10.9 — Pont : seuil, sigmoïde, tanh et ReLU appliqués à un même score linéaire**

In [ ]:
# Ex M10.9: the values you found on paper
answer_M10_9a = ...  # a) a list of 3 numbers
answer_M10_9b = ...  # b) a list of 3 numbers
answer_M10_9c = ...  # c) a list of 3 numbers (4 decimals)
answer_M10_9d = ...  # d) a list of 3 numbers (4 decimals)
answer_M10_9e = ...  # e) a list of 3 numbers
answer_M10_9f = ...  # f) a number (4 decimals)
answer_M10_9g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M10_9a, answer_M10_9b, answer_M10_9c, answer_M10_9d, answer_M10_9e, answer_M10_9f, answer_M10_9g]):
    wbm.check(f"M10.9{letter}", answer)

## Partie A · Ta librairie : enveloppe convexe, séparabilité, marge

Fiche §2 et §3. Les trois fonctions de `mymath/convex.py`.

In [ ]:
# Tools of the notebook (parts A to C)
import inspect

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M10.12 — L'enveloppe convexe dans le plan : la chaîne monotone d'Andrew · `convex_hull_2d` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la chaîne monotone d'Andrew, avec ses cas particuliers : points répétés, alignés, au milieu d'un côté.  
**Prérequis :** fiche §2 · **mymath :** `convex.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/convex.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy est permis, et `scipy.optimize.linprog` pour `is_linearly_separable` ; `scipy.spatial` est interdit (les tests comparent ton enveloppe à `scipy.spatial.ConvexHull`, et un test vérifie que tu ne t'en sers pas). Ne modifie jamais les tableaux reçus.

Écris `convex_hull_2d(points)` (fiche §2), en suivant la docstring : les points distincts triés par $x$ puis par $y$ (`np.unique(P, axis=0)` le fait), la chaîne du bas de gauche à droite, la chaîne du haut de droite à gauche, en retirant le dernier sommet tant que le produit vectoriel est $\le 0$, puis les deux chaînes bout à bout sans répéter leurs extrémités. Le résultat : un tableau NumPy de floats de forme $(h, 2)$, les sommets dans le sens direct à partir du plus petit point. Une `ValueError` si le nuage est vide ou n'a pas deux colonnes.

Avec $P$ = `[[2, -1], [6, 0], [7, 4], [3, 6], [-1, 3], [4, 2], [5, 5], [0, 0], [2, 3], [4, -0.5]]`, la vérification calcule :
- a) `convex_hull_2d(P)` (une liste de points) ;
- b) le nombre de sommets de l'enveloppe de 200 points tirés par `np.random.default_rng(12).normal(size=(200, 2))` ;
- c) `convex_hull_2d([[-2, 3], [4, -3], [0, 1], [2, -1]])` (quatre points alignés) ;
- d) le nom de l'erreur levée par `convex_hull_2d([])` ;
- e) le nom de l'erreur levée par `convex_hull_2d([[1, 2, 3], [4, 5, 6]])` ;
- f) le nom du type de la valeur rendue, en texte.

Dans ta copie : dessine $P$ et son enveloppe. Quel point de $P$ est au milieu d'un côté ? Quel test de ta boucle l'écarte ? Que rendrait ta fonction si le test était $< 0$ au lieu de $\le 0$ ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M10")  # reload your saved file

In [ ]:
P_12 = [[2, -1], [6, 0], [7, 4], [3, 6], [-1, 3], [4, 2], [5, 5], [0, 0], [2, 3], [4, -0.5]]
with checked("M10.12a"):
    wbm.check("M10.12a", np.asarray(mymath.convex.convex_hull_2d(P_12)).tolist(), computed=True)
with checked("M10.12b"):
    cloud_12 = np.random.default_rng(12).normal(size=(200, 2))
    wbm.check("M10.12b", len(mymath.convex.convex_hull_2d(cloud_12)), computed=True)
with checked("M10.12c"):
    wbm.check("M10.12c", np.asarray(mymath.convex.convex_hull_2d([[-2, 3], [4, -3], [0, 1], [2, -1]])).tolist(), computed=True)
with checked("M10.12d"):
    wbm.check("M10.12d", raised(mymath.convex.convex_hull_2d, []), computed=True)
with checked("M10.12e"):
    wbm.check("M10.12e", raised(mymath.convex.convex_hull_2d, [[1, 2, 3], [4, 5, 6]]), computed=True)
with checked("M10.12f"):
    wbm.check("M10.12f", type(mymath.convex.convex_hull_2d(P_12)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m10_convex.py", "convex_hull_2d or test_convex_uses_")

### Ex M10.13 — Séparabilité par programmation linéaire, et marge géométrique · `is_linearly_separable, margin_of_separator` 🔨 ★★ ⏱️ 25 min
**Objectif :** décider la séparabilité par un programme linéaire de faisabilité, et calculer la marge géométrique d'un séparateur.  
**Prérequis :** fiche §3 ; M07 (distance à un hyperplan) · **mymath :** `convex.py` · **Parcours :** C, E

Écris, en suivant les docstrings (fiche §3) :
- `is_linearly_separable(X, y)` : `True` si et seulement si le système $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \ge 1$ a une solution $(\mathbf{w}, b)$, décidé par `scipy.optimize.linprog` (objectif nul, contraintes $-y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \le -1$ écrites ligne par ligne, inconnues libres : `bounds=(None, None)`) ; un `bool` de Python ; une `ValueError` si les longueurs diffèrent ou si un label n'est ni $-1$ ni $+1$ ;
- `margin_of_separator(X, y, w, b)` : $\min_i \frac{y_i(\mathbf{w} \cdot \mathbf{x}_i + b)}{\lVert \mathbf{w} \rVert}$, un `float` de Python ; une `ValueError` si les longueurs ne vont pas, si un label n'est ni $-1$ ni $+1$, ou si $\mathbf{w}$ est nul.

Avec $\mathbf{X}$ = `[[1, 5], [2, 7], [4, 6], [3, 4], [5, 1], [7, 2], [6, -1], [8, 0]]` et $\mathbf{y}$ = `[1, 1, 1, 1, -1, -1, -1, -1]`, la vérification calcule :
- a) `is_linearly_separable(X, y)` ;
- b) la même chose après avoir ajouté le point $(2{,}5 ;\ 5{,}5)$ avec le label $-1$ ;
- c) `is_linearly_separable([[0], [1], [2], [3]], [-1, 1, 1, -1])` ;
- d) `margin_of_separator(X, y, [-1, 1], 1)` (4 décimales) ;
- e) `margin_of_separator(X, y, [2, -2], -2)` (4 décimales) ;
- f) le nom de l'erreur levée par `margin_of_separator(X, y, [0, 0], 1)` ;
- g) le nom de l'erreur levée par `is_linearly_separable([[0, 0], [1, 1]], [0, 1])` ;
- h) vrai ou faux : `is_linearly_separable` rend un `bool` de Python et `margin_of_separator` un `float` de Python (`type(r) is bool and type(m) is float`).

Dans ta copie : pourquoi b) est-il faux, sans aucun calcul de droite (fiche §3) ? Écris d) à la main : quel point est le plus proche de la droite $-x_1 + x_2 + 1 = 0$ ? Pourquoi e) est-il négatif, et pourquoi n'est-il pas l'opposé de d) ? Que rendrait `is_linearly_separable` avec des contraintes $\ge 0$ au lieu de $\ge 1$ ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M10")  # reload your saved file

In [ ]:
X_13, y_13 = [[1, 5], [2, 7], [4, 6], [3, 4], [5, 1], [7, 2], [6, -1], [8, 0]], [1, 1, 1, 1, -1, -1, -1, -1]
with checked("M10.13a"):
    wbm.check("M10.13a", mymath.convex.is_linearly_separable(X_13, y_13), computed=True)
with checked("M10.13b"):
    wbm.check("M10.13b", mymath.convex.is_linearly_separable(X_13 + [[2.5, 5.5]], y_13 + [-1]), computed=True)
with checked("M10.13c"):
    wbm.check("M10.13c", mymath.convex.is_linearly_separable([[0], [1], [2], [3]], [-1, 1, 1, -1]), computed=True)
with checked("M10.13d"):
    wbm.check("M10.13d", mymath.convex.margin_of_separator(X_13, y_13, [-1, 1], 1), computed=True)
with checked("M10.13e"):
    wbm.check("M10.13e", mymath.convex.margin_of_separator(X_13, y_13, [2, -2], -2), computed=True)
with checked("M10.13f"):
    wbm.check("M10.13f", raised(mymath.convex.margin_of_separator, X_13, y_13, [0, 0], 1), computed=True)
with checked("M10.13g"):
    wbm.check("M10.13g", raised(mymath.convex.is_linearly_separable, [[0, 0], [1, 1]], [0, 1]), computed=True)
with checked("M10.13h"):
    wbm.check("M10.13h", type(mymath.convex.is_linearly_separable(X_13, y_13)) is bool
              and type(mymath.convex.margin_of_separator(X_13, y_13, [-1, 1], 1)) is float, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m10_convex.py", "is_linearly_separable or margin_of_separator or test_convex_uses_")

## Partie B · Une prédiction et une figure

Fiche §2 et §3. La séparabilité de labels tirés au hasard, puis tout le §3 sur une seule figure.

### Ex M10.14 — Prédire : n points aux labels tirés au hasard en dimension d sont-ils séparables ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir à quelle vitesse la séparabilité devient rare quand le nombre de points grandit devant la dimension, puis le mesurer.  
**Prérequis :** fiche §3 · **Parcours :** C, E

On tire $n$ points au hasard dans $\mathbb{R}^d$ (des coordonnées gaussiennes : presque sûrement en position générale), et on donne à chacun un label $-1$ ou $+1$ à pile ou face. Les deux classes sont-elles linéairement séparables ? La fiche (§3) dit que $d + 1$ points le sont toujours. Pour chaque cas, choisis le numéro de la bonne réponse : la proportion des tirages séparables est (1) 100 % ; (2) entre 60 % et 95 % ; (3) entre 35 % et 65 % ; (4) moins de 5 % :
- a) $n = 4$ points dans le plan ($d = 2$) ;
- b) $n = 6$ points dans le plan ;
- c) $n = 22$ points en dimension $d = 10$ ;
- d) $n = 30$ points dans le plan.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience : 300 tirages par cas, décidés par la référence de `is_linearly_separable`.

In [ ]:
reference_14 = wbm.load_mymath("ref")     # the experiment uses the reference of convex.py


def separable_fraction(n, d, trials=300, seed=14):
    """The fraction of `trials` draws of n Gaussian points of R^d, with labels -1 or +1 drawn at random, that are
    linearly separable."""
    rng = np.random.default_rng(seed)
    count = 0
    for _ in range(trials):
        X = rng.normal(size=(n, d))
        y = rng.choice([-1, 1], size=n)
        count += reference_14.convex.is_linearly_separable(X, y)
    return count / trials

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M10_14a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M10_14b = ...  # b) 1, 2, 3 or 4
prediction_M10_14c = ...  # c) 1, 2, 3 or 4
prediction_M10_14d = ...  # d) 1, 2, 3 or 4

In [ ]:
for letter, prediction in zip("abcd", [prediction_M10_14a, prediction_M10_14b, prediction_M10_14c,
                                         prediction_M10_14d]):
    wbm.check(f"M10.14{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M10_14a, prediction_M10_14b, prediction_M10_14c, prediction_M10_14d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    for letter, (n, d) in zip("abcd", [(4, 2), (6, 2), (22, 10), (30, 2)]):
        print(f"{letter}) n = {n:2d} points, dimension d = {d:2d} : séparables dans {separable_fraction(n, d):.0%} des "
              f"tirages")

**Après l'expérience**, dans ta copie : pour a), dessine quatre points du plan en position générale ; quelles configurations de labels ne sont pas séparables (pense au critère des enveloppes, fiche §3) ? Que remarques-tu en b) et en c), qui ont tous deux $n = 2(d + 1)$ ? Combien de points faut-il, en dimension $d$, pour que des labels au hasard ne soient presque jamais séparables ? Qu'est-ce que cela dit d'un classifieur linéaire qui sépare parfaitement ses données d'entraînement quand $d$ est grand devant $n$ ?

### Ex M10.15 — Deux nuages, leurs enveloppes convexes et un hyperplan séparateur avec sa marge 🎨 ★ ⏱️ 15 min
**Objectif :** dessiner d'un coup tout le §3 : deux nuages, leurs enveloppes convexes, une droite séparatrice et la bande de sa marge.  
**Prérequis :** fiche §2 et §3 ; Ex M10.12 et M10.13 · **Parcours :** C

Écris `draw_15(ax, X, y, w, b)`, qui dessine dans l'axe `ax` :
- les points, avec `ax.scatter`, une couleur par classe ;
- l'enveloppe convexe de chaque classe (calculée par `mymath.convex.convex_hull_2d`), comme un **polygone fermé** : `ax.plot` sur ses sommets, en répétant le premier à la fin ;
- la droite $\mathbf{w} \cdot \mathbf{x} + b = 0$ et les deux droites parallèles à la distance $\gamma$ de part et d'autre, où $\gamma$ est la marge de $(\mathbf{w}, b)$ (`mymath.convex.margin_of_separator`) : chacune avec `ax.plot`, sur deux points ou plus (sur la droite $\mathbf{w} \cdot \mathbf{x} + b = c$, résous en $x_2$ si $w_2 \ne 0$).

La cellule de la figure s'en sert sur les données de M10.13 ; une cellule de vérification l'essaie sur d'autres données, dans des axes neufs.

Dans ta copie : quels points touchent les bords de la bande ? Pourquoi la bande ne contient-elle aucun point ? Déplace la droite sans changer sa direction pour élargir la bande : jusqu'où peux-tu aller (M10.3 e) ?

In [ ]:
X_15 = np.array([[1, 5], [2, 7], [4, 6], [3, 4], [5, 1], [7, 2], [6, -1], [8, 0]], dtype=float)
y_15 = np.array([1, 1, 1, 1, -1, -1, -1, -1])
w_15, b_15 = np.array([-1.0, 1.0]), 1.0                  # a separator of the two clouds (the one of M10.13 d)

In [ ]:
def draw_15(ax, X, y, w, b):
    """In ax: the two clouds (one color per class), their convex hulls as closed polygons, the line w · x + b = 0 and
    the two parallel lines at the distance of the margin, on each side."""
    raise NotImplementedError("ax.scatter ; convex_hull_2d et ax.plot fermé ; la marge, puis trois droites")

In [ ]:
with checked("M10.15 figure"):
    fig, ax = plt.subplots(figsize=(6, 5))
    try:
        created = drawn(draw_15, ax, X_15, y_15, w_15, b_15)
    except BaseException:
        plt.close(fig)
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M10.15 : draw_15 ouvre sa propre figure : dessine dans l'axe reçu.")
    else:
        ax.set_aspect("equal")
        plt.show()

**Vérifications automatiques** (sur d'autres données, dans des axes neufs).

In [ ]:
with checked("M10.15 vérifications"):
    from matplotlib.patches import Polygon as PolygonPatch
    reference_15 = wbm.load_mymath("ref")
    problems = []
    X_t = np.array([[0.0, 4.0], [1.0, 6.0], [3.0, 5.0], [2.0, 5.0], [4.0, 1.0], [6.0, 2.0], [5.0, -1.0]])   # other data
    y_t = np.array([1, 1, 1, 1, -1, -1, -1])
    w_t, b_t = np.array([-1.0, 2.0]), -3.0
    gamma_t = reference_15.convex.margin_of_separator(X_t, y_t, w_t, b_t)
    fig_t, ax_t = plt.subplots()
    try:
        created = drawn(draw_15, ax_t, X_t, y_t, w_t, b_t)
    finally:
        plt.close(fig_t)
    if created:
        problems.append("draw_15 ouvre sa propre figure : dessine dans l'axe reçu")
    else:
        rings = [np.asarray(line.get_xydata(), dtype=float) for line in ax_t.get_lines()]
        rings += [np.asarray(patch.get_xy(), dtype=float) for patch in ax_t.patches if isinstance(patch, PolygonPatch)]
        for label, name in ((1, "+1"), (-1, "−1")):
            hull_t = reference_15.convex.convex_hull_2d(X_t[y_t == label])
            wanted = {tuple(np.round(p, 6)) for p in hull_t}
            if not any(len(r) >= 2 and np.allclose(r[0], r[-1]) and {tuple(np.round(p, 6)) for p in r} == wanted
                       for r in rings):
                problems.append(f"l'enveloppe de la classe {name} doit être un polygone fermé dont les sommets sont "
                                f"ceux de convex_hull_2d (ax.plot en répétant le premier sommet à la fin)")
        norm_t = float(np.linalg.norm(w_t))
        levels = []
        for line in ax_t.get_lines():
            xy = np.asarray(line.get_xydata(), dtype=float)
            if len(xy) >= 2 and np.ptp(xy[:, 0]) + np.ptp(xy[:, 1]) > 0:
                distances = (xy @ w_t + b_t) / norm_t
                if np.ptp(distances) < 1e-6:
                    levels.append(float(distances[0]))
        for target, what in ((0.0, "la droite séparatrice w · x + b = 0"),
                             (gamma_t, "le bord de la marge du côté +1, à la distance γ"),
                             (-gamma_t, "le bord de la marge du côté −1, à la distance γ")):
            if not any(abs(level - target) < 1e-6 for level in levels):
                problems.append(f"il manque {what} (une droite tracée avec ax.plot)")
        offsets = [np.asarray(c.get_offsets(), dtype=float) for c in ax_t.collections if len(c.get_offsets())]
        shown = {tuple(np.round(p, 6)) for o in offsets for p in o}
        if not {tuple(np.round(p, 6)) for p in X_t} <= shown:
            problems.append("dessine tous les points (ax.scatter), une couleur par classe")
    if problems:
        print("❌ Ex M10.15 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M10.15 : les points, les deux enveloppes, la droite et les deux bords de la marge sont à leur "
              "place.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §2 et §3. Deux erreurs à trouver, des tests de propriétés, puis la plus grande marge.

### Ex M10.16 — Chasse à l'erreur : un produit vectoriel de signe inversé, une marge non normalisée 🐛 ★ ⏱️ 10 min
**Objectif :** repérer dans du code les deux erreurs classiques de ce chapitre, et calculer ce qu'elles font.  
**Prérequis :** fiche §2 et §3 · **Parcours :** C

Un collègue a écrit le code ci-dessous, ligne par ligne commentée. Deux lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter de code**, trouve-les.

```python
def cross(o, a, b):
    return (a[1] - o[1]) * (b[0] - o[0]) - (a[0] - o[0]) * (b[1] - o[1])   # (A) the cross product (a - o) × (b - o)

def half_hull(points):
    chain = []
    for p in points:
        while len(chain) >= 2 and cross(chain[-2], chain[-1], p) <= 0:     # (B) drop the last vertex unless left turn
            chain.pop()
        chain.append(p)
    return chain

def hull(points):
    pts = sorted(set(map(tuple, points)))                                   # (C) distinct points, by x then by y
    lower, upper = half_hull(pts), half_hull(pts[::-1])
    return lower[:-1] + upper[:-1]                                          # (D) the vertices, counterclockwise

def margin(X, y, w, b):
    scores = [yi * (np.dot(w, xi) + b) for xi, yi in zip(X, y)]             # (E) y_i (w · x_i + b) for each point
    return min(scores)                                                      # (F) the geometric margin
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) ce que rend `hull([[1, 0], [4, 1], [3, 4], [0, 3], [2, 2]])` avec ce code **tel quel** (une liste de points) ;
- c) ce que rend `margin([[1, 2], [3, 3], [0, -1], [2, -2]], [1, 1, -1, -1], [1, 2], -1)` avec ce code tel quel ;
- d) la vraie marge géométrique de ce séparateur (4 décimales).

Dans ta copie : corrige les deux lignes. Pourquoi b) donne-t-il quand même les bons sommets ? Quel test de M10.12 attrape la ligne (A), et lequel de M10.13 attrape la ligne (F) ?

In [ ]:
answer_M10_16a = ...  # a) letters, as text
answer_M10_16b = ...  # b) a list of points
answer_M10_16c = ...  # c) a number
answer_M10_16d = ...  # d) a number (4 decimals)

In [ ]:
wbm.check("M10.16a", answer_M10_16a)
wbm.check("M10.16b", answer_M10_16b)
wbm.check("M10.16c", answer_M10_16c)
wbm.check("M10.16d", answer_M10_16d)

### Ex M10.17 — Des tests de propriétés pour l'enveloppe convexe 🛠️ ★ ⏱️ 15 min
**Objectif :** vérifier une enveloppe convexe sans en connaître la réponse, par ses propriétés, comme le font les tests automatiques.  
**Prérequis :** fiche §2 ; Ex M10.12 · **Parcours :** C

Pour tester une fonction sans connaître la réponse exacte, on vérifie des **propriétés** que toute bonne réponse doit avoir (*property-based testing*). Écris `hull_problems(points, hull)`, qui rend la liste des noms des propriétés **non** respectées par `hull`, dans cet ordre, parmi :
- `"sommets"` : chaque sommet de `hull` est l'un des `points` ;
- `"gauche"` : à chaque sommet, le chemin tourne **strictement** à gauche (produit vectoriel $> 0$, en faisant le tour : le dernier sommet suivi du premier) ;
- `"dedans"` : chaque point est à gauche de chaque côté, ou dessus (produit vectoriel $\ge -10^{-9}$, fiche §2) ;
- `"depart"` : le premier sommet est le plus petit point (le plus petit $x$, puis le plus petit $y$).

Une enveloppe juste donne `[]`. On suppose que l'enveloppe a au moins trois sommets.

La cellule de vérification essaie ta fonction sur 20 enveloppes justes (`[]` attendu à chaque fois), puis sur cinq enveloppes faussées : chacune doit être prise en défaut, et pour la bonne raison.

Dans ta copie : pourquoi une enveloppe tournée dans le sens des aiguilles d'une montre échoue-t-elle à **deux** propriétés ? Laquelle de ces propriétés attrape un point au milieu d'un côté ? Une enveloppe qui passe tes quatre tests est-elle forcément juste ? (Pense à un polygone qui oublie un sommet très proche d'un côté.)

In [ ]:
def hull_problems(points, hull):
    """The names of the properties that hull breaks, in the order "sommets", "gauche", "dedans", "depart" ([] for a
    correct convex hull of points, given counterclockwise from the smallest point)."""
    raise NotImplementedError("un test par propriété ; le produit vectoriel ; le tour complet avec le modulo")

In [ ]:
with checked("M10.17 vérifications"):
    reference_17 = wbm.load_mymath("ref")
    problems = []
    for seed in range(1700, 1720):                       # correct hulls: no problem may be reported
        rng_t = np.random.default_rng(seed)
        cloud = rng_t.normal(size=(30, 2)) if seed % 2 else rng_t.integers(0, 6, size=(40, 2)).astype(float)
        found = hull_problems(cloud, reference_17.convex.convex_hull_2d(cloud))
        if list(found) != []:
            problems.append(f"sur une enveloppe juste (graine {seed}), hull_problems rend {list(found)} au lieu de []")
            break
    grid_t = np.array([[0, 0], [2, 0], [4, 0], [4, 3], [2, 4], [0, 3], [1, 1], [2, 2], [3, 1]], dtype=float)
    good_t = reference_17.convex.convex_hull_2d(grid_t)                # [[0,0],[4,0],[4,3],[2,4],[0,3]]
    broken = {
        "dans le sens des aiguilles d'une montre": (np.vstack([good_t[:1], good_t[:0:-1]]), ["gauche", "dedans"]),
        "un sommet oublié": (np.delete(good_t, 2, axis=0), ["dedans"]),
        "le point (2, 0), au milieu d'un côté, ajouté comme sommet": (np.insert(good_t, 1, [2.0, 0.0], axis=0),
                                                                     ["gauche"]),
        "un départ ailleurs qu'au plus petit point": (np.roll(good_t, -1, axis=0), ["depart"]),
        "un sommet poussé hors du nuage": (np.where(np.arange(5)[:, None] == 2, [[4.5, 3.25]], good_t), ["sommets"]),
    }
    for what, (hull_t, expected) in broken.items():
        found = list(hull_problems(grid_t, hull_t))
        if found != expected:
            problems.append(f"pour une enveloppe avec {what}, hull_problems rend {found} au lieu de {expected}")
    if problems:
        print("❌ Ex M10.17 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M10.17 : tes tests de propriétés acceptent les 20 enveloppes justes et attrapent les cinq fausses, "
              "chacune pour la bonne raison.")

### Ex M10.18 — Défi : la plus grande marge possible entre deux nuages donnés, à 1 % près 🏆 ★★★ ⏱️ 20 min
**Objectif :** trouver une droite séparatrice dont la marge est à 1 % de la plus grande possible, sur trois paires de nuages.  
**Prérequis :** fiche §2 et §3 ; Ex M10.12 et M10.13 · **Parcours :** C

Trois datasets du plan, chacun fait de deux nuages séparables de 25 points (`DATASETS_18`, une liste de couples `(X, y)`) ; `BEST_18` donne la plus grande marge possible de chacun. **L'objectif** : écrire `best_separator_18(X, y)`, qui rend un couple `(w, b)` (un vecteur de 2 nombres et un nombre) dont la marge est d'au moins 99 % de la plus grande, sur les trois datasets. Sans scikit-learn ; NumPy, SciPy et ta librairie `mymath` sont permis.

Commence par le plus simple : la médiatrice des deux barycentres. Mesure sa marge avec `margin_18`, et compare à `BEST_18`. Puis sers-toi de la fiche (§3) : à direction fixée, le meilleur décalage $b$ est connu ; et la plus grande marge a une description géométrique. La vérification affiche le rapport de ta marge à la plus grande, pour chaque dataset.

Dans ta copie : quelle marge donnait la médiatrice des barycentres, et pourquoi est-elle si loin du compte ? Quelle méthode as-tu choisie ? Combien de points touchent les bords de la bande de ta meilleure droite ?

In [ ]:
def clouds_18(seed):
    """Two separable clouds of 25 points each in the plane (labels +1 and -1), from a seed."""
    rng = np.random.default_rng(seed)
    angle = rng.uniform(0, 2 * np.pi)
    u = np.array([np.cos(angle), np.sin(angle)])
    A = rng.normal(size=(25, 2)) * [1.6, 0.9] + 3.2 * u
    B = rng.normal(size=(25, 2)) * [1.0, 1.5] - 3.2 * u
    return np.round(np.vstack([A, B]), 2), np.array([1] * 25 + [-1] * 25)


DATASETS_18 = [clouds_18(seed) for seed in (1801, 1802, 1803)]
BEST_18 = [1.316342, 1.028871, 0.96]      # the largest possible margin of each dataset


def margin_18(X, y, w, b):
    """The geometric margin min_i y_i (w · x_i + b) / ‖w‖ (computed here with NumPy, for the check)."""
    w = np.asarray(w, dtype=float).ravel()
    return float(np.min(y * (X @ w + float(b))) / np.linalg.norm(w))

In [ ]:
def best_separator_18(X, y):
    """A couple (w, b) whose geometric margin on (X, y) is at least 99 % of the largest possible."""
    raise NotImplementedError("une direction w, le meilleur b pour elle ; ou les deux enveloppes et leurs points les plus "
                              "proches")

In [ ]:
with checked("M10.18 défi"):
    ok_18 = False
    source_18 = inspect.getsource(best_separator_18)
    if "sklearn" in source_18 or "SVC" in source_18:
        print("❌ Ex M10.18 : le défi se fait sans scikit-learn : la fiche §3 donne de quoi trouver la meilleure droite.")
    else:
        ratios = []
        for (X, y), best in zip(DATASETS_18, BEST_18):
            w, b = best_separator_18(X, y)
            ratios.append(margin_18(X, y, w, b) / best)
        print("ta marge / la plus grande marge :", ", ".join(f"{r:.4f}" for r in ratios))
        if min(ratios) >= 0.99:
            ok_18 = True
            print("✅ Ex M10.18 : défi réussi : à 1 % de la plus grande marge sur les trois datasets.")
        else:
            print("❌ Ex M10.18 : au moins un dataset est à plus de 1 % de la plus grande marge (ou ta droite ne "
                  "sépare pas : un rapport négatif).")
with checked("M10.18"):
    best_separator_18(*DATASETS_18[0])                    # ⏳ while best_separator_18 is not written
    wbm.check("M10.18", bool(ok_18), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu reconnaître un ensemble convexe, et démontrer qu'un demi-espace et une intersection de convexes le sont ?
2. Sais-tu construire l'enveloppe convexe d'un nuage du plan, à la main et en code ?
3. Sais-tu prouver que deux nuages ne sont pas séparables, et le décider par un programme linéaire ?
4. Sais-tu calculer la marge d'un séparateur et la plus grande marge possible, et réécrire un hyperplan avec l'astuce du biais ?
5. Sais-tu borner une norme pas à pas, et opposer une croissance linéaire à une croissance en √k ?

**La suite** : le chapitre 10 de workbookIA (le neurone, du perceptron au neurone moderne), puis M11, la logique, l'induction et les inégalités de concentration.